# 305-02 · JOIN vs EXISTS vs IN: ¿cuál es más rápido?

> **300-Lab Experiment** | Edgar Cartolari Esteves | [github.com/ecartolariesteves/300-Lab](https://github.com/ecartolariesteves/300-Lab)

Tres formas de filtrar por existencia en SQL. Todos los data engineers las usan. Pocos saben cuándo cada una gana.

**TL;DR:** IN gana en filtrado simple. NOT EXISTS es el rey del anti-join. EXISTS puede explotar en filtros multi-condición.

## 0. Setup y datos sintéticos

In [ ]:
import pandas as pd
import numpy as np
import sqlite3
import time
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from IPython.display import display

np.random.seed(42)
N_ORDERS    = 500_000
N_CUSTOMERS = 10_000
N_VIP       = 1_000

customers = pd.DataFrame({
    'customer_id': range(1, N_CUSTOMERS + 1),
    'region':  np.random.choice(['North','South','East','West'], N_CUSTOMERS),
    'segment': np.random.choice(['Premium','Standard','Budget'], N_CUSTOMERS)
})
orders = pd.DataFrame({
    'order_id':    range(1, N_ORDERS + 1),
    'customer_id': np.random.randint(1, N_CUSTOMERS + 1, N_ORDERS),
    'amount':      np.round(np.random.exponential(scale=150, size=N_ORDERS), 2),
})
top_ids = orders.groupby('customer_id')['amount'].sum().nlargest(N_VIP).index.tolist()
vip = pd.DataFrame({'customer_id': top_ids})

conn = sqlite3.connect(':memory:')
customers.to_sql('customers',     conn, index=False, if_exists='replace')
orders.to_sql('orders',           conn, index=False, if_exists='replace')
vip.to_sql('vip_customers',       conn, index=False, if_exists='replace')
conn.execute('CREATE INDEX idx_o_cid ON orders(customer_id)')
conn.execute('CREATE INDEX idx_c_id  ON customers(customer_id)')
conn.execute('CREATE INDEX idx_v_id  ON vip_customers(customer_id)')
conn.commit()
print(f'orders: {len(orders):,} | customers: {len(customers):,} | vip: {len(vip):,}')

In [ ]:
def benchmark(conn, query, runs=7, label=''):
    times = []
    for _ in range(runs):
        t0 = time.perf_counter()
        result = pd.read_sql_query(query, conn)
        times.append((time.perf_counter() - t0) * 1000)
    return {
        'label':   label,
        'mean_ms': round(np.mean(times), 1),
        'min_ms':  round(min(times), 1),
        'rows':    len(result)
    }

all_results = []
print('Benchmark ready ✓')

## 1. Case 1 — Filtrar órdenes de clientes VIP

In [ ]:
q_join = "SELECT o.order_id, o.customer_id, o.amount FROM orders o INNER JOIN vip_customers v ON o.customer_id = v.customer_id"
q_exists = "SELECT o.order_id, o.customer_id, o.amount FROM orders o WHERE EXISTS (SELECT 1 FROM vip_customers v WHERE v.customer_id = o.customer_id)"
q_in = "SELECT o.order_id, o.customer_id, o.amount FROM orders o WHERE o.customer_id IN (SELECT customer_id FROM vip_customers)"

r1j = benchmark(conn, q_join,   label='JOIN')
r1e = benchmark(conn, q_exists, label='EXISTS')
r1i = benchmark(conn, q_in,     label='IN')
all_results.append(('Case 1 — Filter VIP orders', r1j, r1e, r1i))

print(f"JOIN:   {r1j['mean_ms']}ms  ({r1j['rows']:,} rows)")
print(f"EXISTS: {r1e['mean_ms']}ms")
print(f"IN:     {r1i['mean_ms']}ms  ← WINNER")

## 2. Case 2 — COUNT por cliente VIP

In [ ]:
q_join = "SELECT o.customer_id, COUNT(*) AS cnt FROM orders o INNER JOIN vip_customers v ON o.customer_id = v.customer_id GROUP BY o.customer_id"
q_exists = "SELECT o.customer_id, COUNT(*) AS cnt FROM orders o WHERE EXISTS (SELECT 1 FROM vip_customers v WHERE v.customer_id = o.customer_id) GROUP BY o.customer_id"
q_in = "SELECT o.customer_id, COUNT(*) AS cnt FROM orders o WHERE o.customer_id IN (SELECT customer_id FROM vip_customers) GROUP BY o.customer_id"

r2j = benchmark(conn, q_join,   label='JOIN')
r2e = benchmark(conn, q_exists, label='EXISTS')
r2i = benchmark(conn, q_in,     label='IN')
all_results.append(('Case 2 — COUNT per VIP', r2j, r2e, r2i))

factor = round(r2j['mean_ms'] / r2i['mean_ms'], 1)
print(f"JOIN:   {r2j['mean_ms']}ms")
print(f"EXISTS: {r2e['mean_ms']}ms")
print(f"IN:     {r2i['mean_ms']}ms  ← WINNER ({factor}x faster than JOIN)")

## 3. Case 3 — Anti-join: clientes SIN órdenes

> ⚠️ **NOT IN gotcha:** si la subquery contiene algún NULL, NOT IN devuelve 0 filas. Siempre. Es un bug silencioso devastador.

In [ ]:
q_join   = "SELECT c.customer_id FROM customers c LEFT JOIN orders o ON c.customer_id = o.customer_id WHERE o.customer_id IS NULL"
q_exists = "SELECT c.customer_id FROM customers c WHERE NOT EXISTS (SELECT 1 FROM orders o WHERE o.customer_id = c.customer_id)"
q_in     = "SELECT c.customer_id FROM customers c WHERE c.customer_id NOT IN (SELECT DISTINCT customer_id FROM orders)"

r3j = benchmark(conn, q_join,   label='LEFT JOIN + IS NULL')
r3e = benchmark(conn, q_exists, label='NOT EXISTS')
r3i = benchmark(conn, q_in,     label='NOT IN')
all_results.append(('Case 3 — Anti-join', r3j, r3e, r3i))

print(f"LEFT JOIN + IS NULL: {r3j['mean_ms']}ms  ({r3j['rows']} rows with no orders)")
print(f"NOT EXISTS:          {r3e['mean_ms']}ms  ← WINNER (correct + fast)")
print(f"NOT IN:              {r3i['mean_ms']}ms  (works here but dangerous with NULLs)")

# Demonstrate the NULL trap
print("\n--- NULL trap demo ---")
conn.execute("INSERT INTO orders VALUES (999999, NULL, 0.0)")
trap_result = pd.read_sql_query(q_in, conn)
print(f"NOT IN with NULL in subquery: {len(trap_result)} rows returned (should be {r3i['rows']})")
conn.execute("DELETE FROM orders WHERE order_id = 999999")
conn.commit()

## 4. Case 4 — Filtro multi-condición

In [ ]:
q_join   = "SELECT o.order_id, o.amount FROM orders o INNER JOIN customers c ON o.customer_id = c.customer_id WHERE c.segment = 'Premium' AND c.region = 'North'"
q_exists = "SELECT o.order_id, o.amount FROM orders o WHERE EXISTS (SELECT 1 FROM customers c WHERE c.customer_id = o.customer_id AND c.segment = 'Premium' AND c.region = 'North')"
q_in     = "SELECT o.order_id, o.amount FROM orders o WHERE o.customer_id IN (SELECT customer_id FROM customers WHERE segment = 'Premium' AND region = 'North')"

r4j = benchmark(conn, q_join,   label='JOIN')
r4e = benchmark(conn, q_exists, label='EXISTS')
r4i = benchmark(conn, q_in,     label='IN')
all_results.append(('Case 4 — Multi-condition', r4j, r4e, r4i))

print(f"JOIN:   {r4j['mean_ms']}ms  ← WINNER (tied with IN)")
print(f"EXISTS: {r4e['mean_ms']}ms  ← WARNING: re-evaluates correlated subquery per row")
print(f"IN:     {r4i['mean_ms']}ms  ← tied with JOIN")

## 5. Visualización de resultados

In [ ]:
BLUE   = '#4A7FC1'
ORANGE = '#D47C1A'
GREEN  = '#2E8B57'

case_labels = ['Case 1\nFilter VIP', 'Case 2\nCOUNT per VIP', 'Case 3\nAnti-join', 'Case 4\nMulti-condition']
join_vals   = [r1j['mean_ms'], r2j['mean_ms'], r3j['mean_ms'], r4j['mean_ms']]
exists_vals = [r1e['mean_ms'], r2e['mean_ms'], r3e['mean_ms'], r4e['mean_ms']]
in_vals     = [r1i['mean_ms'], r2i['mean_ms'], r3i['mean_ms'], r4i['mean_ms']]

x = np.arange(4)
w = 0.26

fig, ax = plt.subplots(figsize=(11, 5))
ax.set_facecolor('#F5F0E8')
fig.patch.set_facecolor('#F5F0E8')

ax.bar(x - w, join_vals,   width=w, label='JOIN',   color=BLUE,   alpha=0.85)
ax.bar(x,     exists_vals, width=w, label='EXISTS', color=ORANGE, alpha=0.85)
ax.bar(x + w, in_vals,     width=w, label='IN',     color=GREEN,  alpha=0.85)

for i, (j, e, n) in enumerate(zip(join_vals, exists_vals, in_vals)):
    for val, offset, col in [(j, -w, BLUE), (e, 0, ORANGE), (n, w, GREEN)]:
        ax.text(i + offset, val + 3, f'{val:.0f}', ha='center', fontsize=8, color=col, fontweight='bold')

ax.set_xticks(x)
ax.set_xticklabels(case_labels, fontsize=10)
ax.set_ylabel('Execution time (ms)', fontsize=10)
ax.set_title('JOIN vs EXISTS vs IN — Benchmark (500K rows)', fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig('results/benchmark_chart.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved to results/benchmark_chart.png')

## 6. Conclusiones

| | Case 1 Filter | Case 2 COUNT | Case 3 Anti-join | Case 4 Multi-cond |
|---|---|---|---|---|
| **JOIN** | ❌ slower | ❌ slower | ❌ slower | ✅ winner |
| **EXISTS** | ❌ slowest | ❌ slow | ✅ **winner** | ❌ slow (393ms!) |
| **IN** | ✅ **winner** | ✅ **winner (10x)** | ⚠️ careful (NULLs) | ✅ winner |

**Regla práctica:**
```
JOIN       → necesitas columnas de ambas tablas / GROUP BY
IN         → filtros simples, legibilidad, listas pequeñas
EXISTS     → checks correlacionados simples
NOT EXISTS → anti-joins siempre (evita el NULL trap de NOT IN)
NOT IN     → solo si garantizas que no hay NULLs en la subquery
```

---

> 📌 [github.com/ecartolariesteves/300-Lab/305-SQL-Lab/join-vs-exists-in](https://github.com/ecartolariesteves/300-Lab/tree/main/305-SQL-Lab/join-vs-exists-in)